# 01 · Shapes, and walking them

Network data in Python's basic shapes, shapes nested in shapes, and the logic that walks them.
You end by seeing that pandas is the same thing, written by someone else, sitting on your disk.

Run cells top to bottom with **Shift+Enter**. Exercises have a stub that says `your turn` and a
check cell below it; checks print `NOT YET` until you write the function.

In [ ]:
from netlab.check import check   # the exercise checker: src/netlab/check.py (you'll read it in 02)
import time

def best_of(fn, repeat=5):
    """Run fn several times and return the fastest run in seconds (the one least disturbed by other work)."""
    runs = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        runs.append(time.perf_counter() - t0)
    return min(runs)

## A. Information is old

A library card catalog is an index from a key (title, author) to a location (a shelf).
The telegraph turned text into pulses on a wire. A telephone switchboard was a lookup table from a
number to a line. Hollerith's punch-card tabulators counted the 1890 US census: rows of records,
aggregated by machine. None of this needed computers; computers made it fast.

Every program you'll write does three things — **search, retrieve, transform** — to data held in a
handful of shapes. This notebook is those shapes.

### Running cells

A cell runs when you press Shift+Enter. The value of the **last line** is displayed automatically;
`print` shows anything, anywhere, explicitly.

In [ ]:
print("print shows this, from any line")
hostname = "core-sw-01"      # assigning displays nothing
hostname                     # the last line's value is displayed

In [ ]:
# This cell fails on purpose
import traceback
try:
    "port " + 443
except Exception:
    traceback.print_exc()    # print the full traceback, but keep the notebook running

**Reading a traceback: start at the bottom.** The last line is the exception type and message:
`TypeError: can only concatenate str (not "int") to str`. The lines above it say *where*, most recent
call last. It's like reading a log backwards from the alert. Here: you can't glue text to a number;
`"port " + str(443)` would work.

## B. The atoms

One value of each basic kind, each a network thing.

In [ ]:
hostname = "core-sw-01"   # text
port = 443                # a whole number
latency_ms = 12.7         # a measurement
is_up = True              # yes/no
last_seen = None          # "no value yet": a device we've never heard from

for value in (hostname, port, latency_ms, is_up, last_seen):
    print(repr(value), "->", type(value).__name__)

In [ ]:
print(int("443") + 1)     # text that looks like a number is still text until you convert it
print("443" + "1")        # + on text joins; + on numbers adds

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

Floats are binary approximations, so `0.1 + 0.2` is not exactly `0.3`. That's why counters and byte
counts are ints: an int is exact. Use floats for measurements like latency, never for things you count.

### Text vs what crosses the wire

In [ ]:
wire = "GET".encode()        # text -> bytes
print(wire)
print(list(wire))            # what a byte really is: a number 0-255
print(bytes([71, 69, 84]).decode())   # and back

This is the telegraph point: text is a human idea; the wire carries numbers. `"GET"` is `71 69 84`
on the wire. In notebook 03 you'll send these exact bytes over a real socket.

### Binary: you already read masks this way

In [ ]:
print(bin(255), bin(192))
print(int("11000000", 2))
print(f"{192:08b}")            # 8 binary digits, zero-padded, the way you'd write an octet

In [ ]:
octet = 0b10110110
mask  = 0b11110000
print(f"{octet:08b} & {mask:08b} = {octet & mask:08b}")   # AND: keep bits set in both — applying a mask
print(f"{octet:08b} | {mask:08b} = {octet | mask:08b}")   # OR: bits set in either
print(1 << 8, 1 << 24)        # shift left: each step doubles (moves bits up)
print(256 >> 1, 0b1000 >> 3)  # shift right: each step halves (moves bits down)

### Exercise 1 — an address is one number

An IPv4 address is four numbers 0–255, but on the wire and in a routing table it's one 32-bit number:
the first octet is the top 8 bits. Write the conversion in both directions. You'll need it in every
exercise that follows.

In [ ]:
def ip_to_int(ip: str) -> int:
    """Turn a dotted IPv4 address into one 32-bit number.

    Input: a string like "10.0.0.1" (four numbers 0-255 separated by dots).
    Output: an int; the first number is the most significant 8 bits.
    Examples: "0.0.0.1" -> 1, "0.0.1.0" -> 256, "1.0.0.0" -> 16777216.
    """
    raise NotImplementedError("your turn")


def int_to_ip(n: int) -> str:
    """The reverse: a number from 0 to 4294967295 back to dotted form."""
    raise NotImplementedError("your turn")

In [ ]:
check(ip_to_int, [
    (("0.0.0.0",), 0),
    (("255.255.255.255",), 4294967295),
    (("10.0.0.1",), 167772161),
    (("192.168.1.10",), 3232235786),
])
check(int_to_ip, [
    ((0,), "0.0.0.0"),
    ((4294967295,), "255.255.255.255"),
    ((167772161,), "10.0.0.1"),
])
check(lambda ip: int_to_ip(ip_to_int(ip)), [
    (("172.16.254.3",), "172.16.254.3"),
    (("8.8.8.8",), "8.8.8.8"),
], name="round trip");

### Exercise 2 — is this address in that subnet?

A host is in `10.20.0.0/16` if its first 16 bits equal the network's first 16 bits. Build a mask from
the prefix length, apply it to both addresses, compare. Use your Exercise 1. Watch the edges: `/32`
is one address, `/0` is every address.

In [ ]:
def in_subnet(ip: str, cidr: str) -> bool:
    """True if ip falls inside the network written in CIDR form.

    Inputs: ip like "192.168.1.77"; cidr like "192.168.1.0/24".
    Output: True or False.
    """
    raise NotImplementedError("your turn")

In [ ]:
check(in_subnet, [
    (("10.1.2.3", "10.1.2.3/32"), True),
    (("10.1.2.4", "10.1.2.3/32"), False),
    (("8.8.8.8", "0.0.0.0/0"), True),
    (("192.168.1.0", "192.168.1.0/24"), True),
    (("192.168.1.255", "192.168.1.0/24"), True),
    (("192.168.2.0", "192.168.1.0/24"), False),
    (("192.168.0.255", "192.168.1.0/24"), False),
    (("10.0.0.0", "10.0.0.0/31"), True),
    (("10.0.0.1", "10.0.0.0/31"), True),
    (("10.0.0.2", "10.0.0.0/31"), False),
]);

In [ ]:
import ipaddress   # part of Python's standard library

print(ipaddress.ip_address("192.168.1.255") in ipaddress.ip_network("192.168.1.0/24"))
print(int(ipaddress.ip_address("10.0.0.1")))
print(ipaddress.ip_address(167772161))

That is what libraries are: someone already wrote your function. Writing it yourself first is how you
know what the library is doing for you, and when to trust it.

<details><summary>What people call this</summary>

Converting between text and a number: <b>parsing</b> and <b>serialization</b>.<br>
<code>&amp;</code>, <code>|</code>, <code>&lt;&lt;</code>, <code>&gt;&gt;</code> are <b>bitwise operators</b>; a mask applied with <code>&amp;</code> is <b>bit masking</b>.<br>
The kinds of value (<code>str</code>, <code>int</code>, <code>float</code>, <code>bool</code>, <code>None</code>) are <b>types</b>; the basic ones are <b>primitive types</b>.
</details>

## C. Holding many things

### In order, by position

A traceroute is a sequence of hops, in order. Position matters.

In [ ]:
path = ["10.0.0.1", "172.16.0.1", "203.0.113.9", "198.51.100.4", "93.184.216.34"]

print(path[0])        # first hop (positions count from 0)
print(path[-1])       # last hop
print(path[1:3])      # a slice: positions 1 and 2
print(len(path))

In [ ]:
path.append("93.184.216.35")   # add to the end
print(path)
print("203.0.113.9" in path)   # search: is this hop on the path?

How long does `in` take? Measure it on a million flow IDs, looking for one near the start, the middle, the end, and one that isn't there.

In [ ]:
big_list = list(range(1_000_000))

for label, target in [("start", 0), ("middle", 500_000), ("end", 999_999), ("missing", -1)]:
    seconds = best_of(lambda: target in big_list)
    print(f"{label:>8}: {seconds * 1e6:10.1f} µs")

The time grows with how far in the item is, and "missing" is the slowest: `in` on this shape
**walks from the front** and checks every item until it finds a match. That's a **search**.

### A fixed record

A flow's 5-tuple: five fields, fixed positions, and it shouldn't change once captured.

In [ ]:
flow = ("10.0.0.5", "93.184.216.34", 51544, 443, "tcp")
src_ip, dst_ip, src_port, dst_port, proto = flow    # unpack by position, like reading header fields
print(dst_port, proto)

In [ ]:
# This cell fails on purpose
try:
    flow[3] = 8443
except Exception as e:
    print(type(e).__name__, "-", e)

### Find by name, instantly

An ARP table maps IP → MAC. A DNS cache maps name → IP. You don't walk them; you ask by key.

In [ ]:
arp = {"10.0.0.1": "00:1a:2b:3c:4d:01", "10.0.0.5": "00:1a:2b:3c:4d:05"}
dns_cache = {"example.com": "93.184.216.34", "github.com": "140.82.112.3"}
services = {22: "ssh", 53: "dns", 80: "http", 443: "https", 3389: "rdp"}

print(arp["10.0.0.5"])               # retrieve by key
arp["10.0.0.9"] = "00:1a:2b:3c:4d:09"  # set (add or overwrite)
print(services.get(8080, "unknown"))   # retrieve with a default when the key is missing
print(len(arp), list(dns_cache))

In [ ]:
# This cell fails on purpose
try:
    arp["10.0.0.99"]
except Exception as e:
    print(type(e).__name__, "-", e)

In [ ]:
big_dict = {i: i for i in range(1_000_000)}

list_end = best_of(lambda: 999_999 in big_list)
dict_end = best_of(lambda: 999_999 in big_dict)
print(f"list, last item:  {list_end * 1e6:10.2f} µs")
print(f"dict, same key:   {dict_end * 1e6:10.2f} µs")
print(f"the dict was about {list_end / dict_end:,.0f}x faster on this machine")

The dict doesn't walk. It computes *where* the key lives from the key itself and goes straight there,
so a million keys costs the same as ten. That is the card catalog. This operation is **retrieve**.

In [ ]:
flow_table = {}                                   # 5-tuple -> byte count
key = ("10.0.0.5", "93.184.216.34", 51544, 443, "tcp")
flow_table[key] = 1532
flow_table[key] += 800                            # the same flow, more bytes
print(flow_table)

In [ ]:
# This cell fails on purpose
try:
    flow_table[["10.0.0.5", 443]] = 1
except Exception as e:
    print(type(e).__name__, "-", e)

A key's location is computed from its contents, so a key must never change after it's filed: a
list can change, so it can't be a key; a tuple can't, so it can.

### Membership and overlap

In [ ]:
blocklist = {"203.0.113.66", "198.51.100.23", "192.0.2.10"}
print("203.0.113.66" in blocklist)        # instant, like a dict key

vlan10  = {"10.0.10.5", "10.0.10.6", "10.0.10.7", "10.0.10.8"}
allowed = {"10.0.10.5", "10.0.10.7", "10.0.20.3"}
seen    = {"10.0.10.5", "10.0.20.3"}

print(vlan10 & allowed)    # in both
print(allowed - seen)      # allowed but never seen
print(vlan10 | allowed)    # in either

### Exercise 3 — parse a firewall log line

Firewall logs arrive as lines of text. Turn one into a dict with typed fields: ports and byte
counts as ints. Real logs are messy: tolerate extra spaces, and return `None` for a line you can't
parse instead of crashing.

In [ ]:
def parse_line(line: str) -> dict | None:
    """Parse one firewall log line.

    Input: "2026-03-01T10:00:02Z ALLOW 10.0.0.5:51544 -> 93.184.216.34:443 tcp 1532"
    Output: {"ts": "2026-03-01T10:00:02Z", "action": "ALLOW",
             "src_ip": "10.0.0.5", "src_port": 51544,
             "dst_ip": "93.184.216.34", "dst_port": 443,
             "proto": "tcp", "bytes": 1532}
    Extra spaces between fields are fine. Return None if the line has the wrong number of
    fields, no "->", an address without ":port", or a port or byte count that isn't a number.
    """
    raise NotImplementedError("your turn")

In [ ]:
check(parse_line, [
    (("2026-03-01T10:00:02Z ALLOW 10.0.0.5:51544 -> 93.184.216.34:443 tcp 1532",),
     {"ts": "2026-03-01T10:00:02Z", "action": "ALLOW", "src_ip": "10.0.0.5", "src_port": 51544,
      "dst_ip": "93.184.216.34", "dst_port": 443, "proto": "tcp", "bytes": 1532}),
    (("2026-03-01T10:00:05Z   DENY  203.0.113.66:4444  ->  10.0.0.20:22 tcp   60",),
     {"ts": "2026-03-01T10:00:05Z", "action": "DENY", "src_ip": "203.0.113.66", "src_port": 4444,
      "dst_ip": "10.0.0.20", "dst_port": 22, "proto": "tcp", "bytes": 60}),
    (("2026-03-01T10:00:03Z ALLOW 10.0.0.7:50122 -> 8.8.8.8:53 udp 74",),
     {"ts": "2026-03-01T10:00:03Z", "action": "ALLOW", "src_ip": "10.0.0.7", "src_port": 50122,
      "dst_ip": "8.8.8.8", "dst_port": 53, "proto": "udp", "bytes": 74}),
    (("this is not a log line",), None),
    (("2026-03-01T10:00:09Z ALLOW 10.0.0.5:abc -> 8.8.8.8:53 udp 80",), None),
    (("2026-03-01T10:00:09Z ALLOW 10.0.0.5 -> 8.8.8.8:53 udp 80",), None),
    (("",), None),
]);

Here are a dozen log lines, and the same lines already parsed: what a correct `parse_line` returns
for each. Later cells use `ROWS`, so nothing below depends on your solution.

In [ ]:
LOG_LINES = [
    '2026-03-01T10:00:02Z ALLOW 10.0.0.5:51544 -> 93.184.216.34:443 tcp 1532',
    '2026-03-01T10:00:03Z ALLOW 10.0.0.7:50122 -> 8.8.8.8:53 udp 74',
    '2026-03-01T10:00:04Z DENY 203.0.113.66:4444 -> 10.0.0.20:22 tcp 60',
    '2026-03-01T10:00:05Z ALLOW 10.0.0.5:51545 -> 93.184.216.34:443 tcp 48210',
    '2026-03-01T10:00:07Z ALLOW 10.0.0.12:49820 -> 151.101.1.69:443 tcp 9120',
    '2026-03-01T10:00:08Z ALLOW 10.0.0.7:50130 -> 8.8.4.4:53 udp 81',
    '2026-03-01T10:00:09Z DENY 198.51.100.23:51000 -> 10.0.0.20:3389 tcp 52',
    '2026-03-01T10:00:11Z ALLOW 10.0.0.12:49822 -> 140.82.112.3:443 tcp 22400',
    '2026-03-01T10:00:12Z ALLOW 10.0.0.30:40001 -> 10.0.0.20:5432 tcp 3300',
    '2026-03-01T10:00:14Z ALLOW 10.0.0.5:51550 -> 104.16.132.229:443 tcp 15000',
    '2026-03-01T10:00:15Z DENY 203.0.113.66:4445 -> 10.0.0.21:22 tcp 60',
    '2026-03-01T10:00:17Z ALLOW 10.0.0.30:40007 -> 10.0.0.20:5432 tcp 2100',
]

In [ ]:
ROWS = [{'ts': '2026-03-01T10:00:02Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.5',
  'src_port': 51544,
  'dst_ip': '93.184.216.34',
  'dst_port': 443,
  'proto': 'tcp',
  'bytes': 1532},
 {'ts': '2026-03-01T10:00:03Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.7',
  'src_port': 50122,
  'dst_ip': '8.8.8.8',
  'dst_port': 53,
  'proto': 'udp',
  'bytes': 74},
 {'ts': '2026-03-01T10:00:04Z',
  'action': 'DENY',
  'src_ip': '203.0.113.66',
  'src_port': 4444,
  'dst_ip': '10.0.0.20',
  'dst_port': 22,
  'proto': 'tcp',
  'bytes': 60},
 {'ts': '2026-03-01T10:00:05Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.5',
  'src_port': 51545,
  'dst_ip': '93.184.216.34',
  'dst_port': 443,
  'proto': 'tcp',
  'bytes': 48210},
 {'ts': '2026-03-01T10:00:07Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.12',
  'src_port': 49820,
  'dst_ip': '151.101.1.69',
  'dst_port': 443,
  'proto': 'tcp',
  'bytes': 9120},
 {'ts': '2026-03-01T10:00:08Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.7',
  'src_port': 50130,
  'dst_ip': '8.8.4.4',
  'dst_port': 53,
  'proto': 'udp',
  'bytes': 81},
 {'ts': '2026-03-01T10:00:09Z',
  'action': 'DENY',
  'src_ip': '198.51.100.23',
  'src_port': 51000,
  'dst_ip': '10.0.0.20',
  'dst_port': 3389,
  'proto': 'tcp',
  'bytes': 52},
 {'ts': '2026-03-01T10:00:11Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.12',
  'src_port': 49822,
  'dst_ip': '140.82.112.3',
  'dst_port': 443,
  'proto': 'tcp',
  'bytes': 22400},
 {'ts': '2026-03-01T10:00:12Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.30',
  'src_port': 40001,
  'dst_ip': '10.0.0.20',
  'dst_port': 5432,
  'proto': 'tcp',
  'bytes': 3300},
 {'ts': '2026-03-01T10:00:14Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.5',
  'src_port': 51550,
  'dst_ip': '104.16.132.229',
  'dst_port': 443,
  'proto': 'tcp',
  'bytes': 15000},
 {'ts': '2026-03-01T10:00:15Z',
  'action': 'DENY',
  'src_ip': '203.0.113.66',
  'src_port': 4445,
  'dst_ip': '10.0.0.21',
  'dst_port': 22,
  'proto': 'tcp',
  'bytes': 60},
 {'ts': '2026-03-01T10:00:17Z',
  'action': 'ALLOW',
  'src_ip': '10.0.0.30',
  'src_port': 40007,
  'dst_ip': '10.0.0.20',
  'dst_port': 5432,
  'proto': 'tcp',
  'bytes': 2100}]

### Exercise 4 — first matching rule wins

A firewall checks rules top to bottom and applies the first one that matches; if none match, it
denies. Each rule has a source `src` (CIDR), a `dst_port` (a number, or `"any"`), and an `action`.
Use your Exercise 2.

In [ ]:
def first_match(rules: list[dict], packet: dict) -> str:
    """Return the action of the first rule that matches the packet, or "deny" if none match.

    rules: [{"src": "10.0.0.0/8", "dst_port": 22, "action": "allow"}, ...] in order.
    packet: {"src_ip": "10.1.1.1", "dst_port": 22}
    A rule matches when the packet's src_ip is inside rule["src"] and the rule's dst_port is
    either "any" or equal to the packet's dst_port.
    """
    raise NotImplementedError("your turn")

In [ ]:
RULES = [
    {"src": "203.0.113.0/24", "dst_port": "any", "action": "deny"},    # a known-bad range
    {"src": "0.0.0.0/0",      "dst_port": 443,   "action": "allow"},   # HTTPS from anywhere
    {"src": "10.0.0.0/8",     "dst_port": 22,    "action": "allow"},   # SSH from inside only
]
RULES_SWAPPED = [RULES[1], RULES[0], RULES[2]]   # same rules, first two swapped

check(first_match, [
    ((RULES, {"src_ip": "8.8.8.8", "dst_port": 443}), "allow"),
    ((RULES, {"src_ip": "203.0.113.5", "dst_port": 443}), "deny"),
    ((RULES, {"src_ip": "10.1.1.1", "dst_port": 22}), "allow"),
    ((RULES, {"src_ip": "8.8.8.8", "dst_port": 22}), "deny"),
    ((RULES, {"src_ip": "10.1.1.1", "dst_port": 3389}), "deny"),
    ((RULES_SWAPPED, {"src_ip": "203.0.113.5", "dst_port": 443}), "allow"),
]);

The last two checks use the same rules and the same packet, and get opposite answers. **Order is
part of the data.** A list keeps order; that's why rules live in one.

### Exercise 5 — the most specific route wins

A routing table is a list of `(cidr, next_hop)`. Several prefixes can match one address; the router
uses the most specific (longest prefix). The default route `0.0.0.0/0` matches everything. Note the
table below is *not* sorted by specificity.

In [ ]:
def route_lookup(ip: str, table: list[tuple[str, str]]) -> str:
    """Return the next hop of the most specific prefix in table that contains ip.

    table: [("0.0.0.0/0", "isp-gw"), ("10.0.0.0/8", "core-rtr-01"), ...] in any order.
    """
    raise NotImplementedError("your turn")

In [ ]:
ROUTES = [
    ("0.0.0.0/0",      "isp-gw"),
    ("10.0.0.0/8",     "core-rtr-01"),
    ("10.20.30.0/24",  "acc-sw-03"),
    ("10.20.0.0/16",   "dist-sw-02"),
    ("192.168.0.0/16", "lab-fw"),
]
check(route_lookup, [
    (("10.20.30.40", ROUTES), "acc-sw-03"),
    (("10.20.30.255", ROUTES), "acc-sw-03"),
    (("10.20.99.1", ROUTES), "dist-sw-02"),
    (("10.99.0.1", ROUTES), "core-rtr-01"),
    (("8.8.8.8", ROUTES), "isp-gw"),
    (("192.168.5.5", ROUTES), "lab-fw"),
    (("10.20.30.40", list(reversed(ROUTES))), "acc-sw-03"),
]);

<details><summary>What people call this</summary>

An ordered, changeable sequence (<code>list</code>) is a <b>dynamic array</b>. Walking it to find something is <b>linear search</b>; its cost grows with the size, written <b>O(n)</b>.<br>
A fixed record (<code>tuple</code>) is <b>immutable</b>.<br>
A <code>dict</code> is a <b>hash map</b> (or hash table). Lookup cost doesn't grow with size: <b>O(1)</b>. Keys must be <b>hashable</b>, which in practice means immutable.<br>
A <code>set</code> is a <b>hash set</b>; <code>&amp;</code>, <code>|</code>, <code>-</code> are <b>intersection</b>, <b>union</b>, <b>difference</b>.<br>
Most-specific-prefix routing is <b>longest prefix match</b>.
</details>

## D. Shapes made of shapes

### Rows vs columns

The same six flows, two ways: one dict per row, or one list per column.

In [ ]:
as_rows = [{"src_ip": r["src_ip"], "dst_port": r["dst_port"], "bytes": r["bytes"]} for r in ROWS[:6]]

as_columns = {
    "src_ip":   [r["src_ip"] for r in as_rows],
    "dst_port": [r["dst_port"] for r in as_rows],
    "bytes":    [r["bytes"] for r in as_rows],
}
print(as_rows[0])
print(as_columns["bytes"])
print(sum(as_columns["bytes"]))   # a column is what you want for "total bytes"

Same information, two layouts. Logs arrive as rows (one event at a time); analysis wants columns
(every value of one field together). Notebook 03 meets this again as CSV (rows) vs Parquet (columns).

### Exercise 6 — convert between the layouts

In [ ]:
def rows_to_columns(rows: list[dict]) -> dict[str, list]:
    """List of row dicts (all with the same keys) -> dict of column lists, in row order.

    [{"a": 1, "b": 2}, {"a": 3, "b": 4}] -> {"a": [1, 3], "b": [2, 4]}
    An empty list gives {}.
    """
    raise NotImplementedError("your turn")


def columns_to_rows(columns: dict[str, list]) -> list[dict]:
    """The reverse. {"a": [1, 3], "b": [2, 4]} -> [{"a": 1, "b": 2}, {"a": 3, "b": 4}]. {} gives []."""
    raise NotImplementedError("your turn")

In [ ]:
check(rows_to_columns, [
    (([{"a": 1, "b": 2}, {"a": 3, "b": 4}],), {"a": [1, 3], "b": [2, 4]}),
    ((as_rows,), as_columns),
    (([],), {}),
])
check(columns_to_rows, [
    (({"a": [1, 3], "b": [2, 4]},), [{"a": 1, "b": 2}, {"a": 3, "b": 4}]),
    ((as_columns,), as_rows),
    (({},), []),
]);

### A topology

A dict mapping each device to the list of devices it links to. Here's the network used for the
rest of this notebook (and served by an API in notebook 03):

```
                [ internet ]
                     |
                 edge-rtr
                     |
                   fw-01
                  /     \
       core-rtr-01 ----- core-rtr-02
            |     \       /     |
            |       \   /       |
            |         X         |
            |       /   \       |
        dist-sw-01         dist-sw-02
         /      \               |
   acc-sw-01  acc-sw-02     acc-sw-03
```

In [ ]:
TOPOLOGY = {
    "edge-rtr":    ["fw-01"],
    "fw-01":       ["edge-rtr", "core-rtr-01", "core-rtr-02"],
    "core-rtr-01": ["fw-01", "core-rtr-02", "dist-sw-01", "dist-sw-02"],
    "core-rtr-02": ["fw-01", "core-rtr-01", "dist-sw-01", "dist-sw-02"],
    "dist-sw-01":  ["core-rtr-01", "core-rtr-02", "acc-sw-01", "acc-sw-02"],
    "dist-sw-02":  ["core-rtr-01", "core-rtr-02", "acc-sw-03"],
    "acc-sw-01":   ["dist-sw-01"],
    "acc-sw-02":   ["dist-sw-01"],
    "acc-sw-03":   ["dist-sw-02"],
}

print(TOPOLOGY["fw-01"])            # neighbors of the firewall
print(len(TOPOLOGY), "devices")

### Headers on headers

A sender builds a tunneled packet by wrapping: payload first, then each header around it. The
receiver unwraps in the reverse order. Adding and removing at the same end is exactly `append` and `pop`.

In [ ]:
stack = []
for header in ["payload", "tcp", "ip(inner)", "gre", "ip(outer)", "eth"]:
    stack.append(header)               # encapsulate: wrap one more layer around
print("built:", stack)

print("outermost first, the way a capture tool shows it:", list(reversed(stack)))

print(stack.pop(), stack.pop())        # decapsulate: the last layer added comes off first
print("left:", stack)

### Exercise 7 — strip the tunnel

Packets below are written **outermost first**, as a capture shows them. Strip headers from the
outside until the one you want is outermost — say, to inspect the inner IP header of a GRE tunnel.

In [ ]:
def decapsulate(packet: list, until: str) -> tuple[list, list]:
    """Remove headers from the front of packet until the header named `until` is first.

    Returns (remaining packet, headers removed in the order they were removed).
    If `until` is not in the packet, everything is removed: ([], all headers).
    Don't change the list you were given; work on a copy.
    """
    raise NotImplementedError("your turn")

In [ ]:
PKT = ["eth", "ip(outer)", "gre", "ip(inner)", "tcp", "payload"]
check(decapsulate, [
    ((PKT, "ip(inner)"), (["ip(inner)", "tcp", "payload"], ["eth", "ip(outer)", "gre"])),
    ((PKT, "eth"), (PKT, [])),
    ((PKT, "payload"), (["payload"], ["eth", "ip(outer)", "gre", "ip(inner)", "tcp"])),
    ((PKT, "udp"), ([], PKT)),
]);

### A waiting line

An interface buffer: packets arrive, wait, and leave in arrival order. Removing from the front of a
list makes Python shift every remaining item forward. `collections.deque` is built to add and remove
at both ends cheaply. Measure the difference draining 100,000 packets.

In [ ]:
from collections import deque

def drain_list():
    q = list(range(100_000))
    while q:
        q.pop(0)

def drain_deque():
    q = deque(range(100_000))
    while q:
        q.popleft()

t_list = best_of(drain_list, repeat=1)
t_deque = best_of(drain_deque, repeat=3)
print(f"list.pop(0):     {t_list:8.3f} s")
print(f"deque.popleft(): {t_deque:8.3f} s")

### Exercise 8 — a tail-drop buffer

Model an interface: `arrivals[t]` packets arrive at tick `t`. Each tick, arrivals are queued first
(any beyond `capacity` are dropped), then up to `service_rate` packets leave. Keep ticking after the
last arrival until the buffer is empty. Use a `deque`.

In [ ]:
def simulate_buffer(arrivals: list[int], service_rate: int, capacity: int) -> dict:
    """Simulate a FIFO interface buffer.

    Each tick t: arrivals[t] packets arrive; each one is queued if the queue holds fewer than
    `capacity` packets, otherwise dropped. Then up to service_rate packets leave (are sent).
    After the last arrival, keep sending until the queue is empty.
    Returns {"sent": int, "dropped": int, "max_depth": int}, where max_depth is the largest
    queue length seen right after arrivals were queued.
    """
    raise NotImplementedError("your turn")

In [ ]:
check(simulate_buffer, [
    (([3, 0, 0], 1, 10), {"sent": 3, "dropped": 0, "max_depth": 3}),
    (([5, 5], 2, 4), {"sent": 6, "dropped": 4, "max_depth": 4}),
    (([2, 2, 2, 2], 2, 3), {"sent": 8, "dropped": 0, "max_depth": 2}),
    (([10], 3, 10), {"sent": 10, "dropped": 0, "max_depth": 10}),
    (([], 1, 5), {"sent": 0, "dropped": 0, "max_depth": 0}),
]);

This is **tail drop**: when the buffer is full, new arrivals are discarded. You now have a model you
can tune: try a burst like `[50, 0, 0, 0]` with different capacities and service rates.

<details><summary>What people call this</summary>

A dict of lists of dicts is <b>nested data</b>. Rows vs columns are <b>row-oriented</b> and <b>column-oriented</b> layouts.<br>
A device → neighbors dict is a <b>graph</b> stored as an <b>adjacency list</b>; devices are <b>nodes</b> (vertices), links are <b>edges</b>.<br>
Add and remove at the same end: a <b>stack</b>, last in first out (<b>LIFO</b>).<br>
Add at one end, remove at the other: a <b>queue</b>, first in first out (<b>FIFO</b>). <code>deque</code> is a <b>double-ended queue</b>.
</details>

## E. Naming your shapes

### The problem with dicts

A dict accepts any key. A typo doesn't fail; it quietly creates a new field.

In [ ]:
device = {"hostname": "core-sw-01", "ip": "10.0.0.1", "role": "core"}
device["hostanme"] = "core-sw-02"     # meant to rename it; typo'd the key
print(device)                          # two names now, and the real one never changed

### A dict with a contract

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class Device:
    name: str
    ip: str
    role: str
    links: list[str] = field(default_factory=list)   # each Device gets its own new list

    def in_subnet(self, cidr: str) -> bool:
        return in_subnet(self.ip, cidr)                # your Exercise 2

fw = Device("fw-01", "10.255.0.2", "firewall", ["edge-rtr", "core-rtr-01"])
print(fw)                                              # a readable printout, for free
print(fw.role, fw.links)
print(fw == Device("fw-01", "10.255.0.2", "firewall", ["edge-rtr", "core-rtr-01"]))   # equal by contents

In [ ]:
# This cell fails on purpose
try:
    Device(name="core-sw-02", ip="10.0.0.2", role="core", hostanme="oops")
except Exception as e:
    print(type(e).__name__, "-", e)

In [ ]:
try:
    print(fw.in_subnet("10.255.0.0/16"))     # behavior attached to the data
except NotImplementedError:
    print("NOT YET (this method calls your in_subnet from Exercise 2)")

print(asdict(fw))                           # back to a plain dict when you need one (e.g. JSON)

In [ ]:
@dataclass(frozen=True)
class FlowKey:
    src_ip: str
    dst_ip: str
    src_port: int
    dst_port: int
    proto: str

flows = {FlowKey("10.0.0.5", "93.184.216.34", 51544, 443, "tcp"): 1532}   # frozen, so it can be a key
k = FlowKey("10.0.0.5", "93.184.216.34", 51544, 443, "tcp")
print(flows[k])

In [ ]:
# This cell fails on purpose
try:
    k.dst_port = 8443
except Exception as e:
    print(type(e).__name__, "-", e)

A dataclass is the same shapes with names and rules attached. It's more flexible than a dict because
it can carry behavior (`in_subnet`), and stricter because it rejects nonsense (a typo'd field fails
loudly instead of silently). `frozen=True` gives you the tuple's guarantee with field names.

Rebuild the topology from §D as named devices:

In [ ]:
DEVICE_INFO = {   # name -> (ip, role)
    "edge-rtr": ("10.255.0.1", "internet-edge"), "fw-01": ("10.255.0.2", "firewall"),
    "core-rtr-01": ("10.0.0.1", "core"),         "core-rtr-02": ("10.0.0.2", "core"),
    "dist-sw-01": ("10.10.0.1", "distribution"), "dist-sw-02": ("10.20.0.1", "distribution"),
    "acc-sw-01": ("10.10.1.1", "access"),        "acc-sw-02": ("10.10.2.1", "access"),
    "acc-sw-03": ("10.20.30.1", "access"),
}
devices = {}
for name, links in TOPOLOGY.items():
    ip, role = DEVICE_INFO[name]
    devices[name] = Device(name, ip, role, links)

print(devices["dist-sw-02"])
print([d.name for d in devices.values() if d.role == "core"])

<details><summary>What people call this</summary>

A <code>@dataclass</code> defines a <b>class</b>; each <code>Device(...)</code> is an <b>instance</b> (an <b>object</b>). Its fields are <b>attributes</b>; a function attached to it is a <b>method</b>.<br>
The field list with types is a <b>schema</b>; <code>str</code>, <code>int</code> in it are <b>type hints</b>.
</details>

## F. Walking the shapes

Every structure so far is useless without logic that walks it. Walking is how you search, retrieve,
and transform: visit each item, decide, accumulate.

### Exercise 9 — top talkers

Which sources sent the most bytes? Walk the rows, accumulate a total per source IP in a dict
(**transform**), then sort the totals largest first: `sorted(items, key=..., reverse=True)`.

In [ ]:
def top_talkers(rows: list[dict], n: int) -> list[tuple[str, int]]:
    """Total bytes per src_ip, largest first, at most n entries.

    rows: parsed log rows like ROWS.
    Output: [("10.0.0.5", 64742), ...]
    """
    raise NotImplementedError("your turn")

In [ ]:
check(top_talkers, [
    ((ROWS, 3), [("10.0.0.5", 64742), ("10.0.0.12", 31520), ("10.0.0.30", 5400)]),
    ((ROWS, 1), [("10.0.0.5", 64742)]),
    ((ROWS[:2], 5), [("10.0.0.5", 1532), ("10.0.0.7", 74)]),
    (([], 3), []),
]);

### Exercise 10 — who can this device reach?

Think of flooding: you tell your neighbors, they tell theirs, and you keep a list of who has already
heard so nobody hears twice. A `deque` holds the devices waiting to tell their neighbors; a `set`
holds who has heard.

In [ ]:
def reachable(topology: dict, start: str) -> set:
    """Every device reachable from start by following links, including start itself."""
    raise NotImplementedError("your turn")

In [ ]:
WITH_ISOLATED = {**TOPOLOGY, "lab-sw-99": []}    # a switch someone forgot to cable
ALL = set(TOPOLOGY)

check(reachable, [
    ((TOPOLOGY, "fw-01"), ALL),
    ((TOPOLOGY, "acc-sw-03"), ALL),
    ((WITH_ISOLATED, "core-rtr-01"), ALL),
    ((WITH_ISOLATED, "lab-sw-99"), {"lab-sw-99"}),
    (({"a": ["b"], "b": ["a"], "c": []}, "a"), {"a", "b"}),
]);

### Exercise 11 — fewest hops

Same flood, but when a device first hears, remember *who told it*. When the flood reaches `dst`,
walk those "who told me" links backwards to `src`, then reverse.

In [ ]:
def fewest_hops(topology: dict, src: str, dst: str) -> list[str] | None:
    """A path from src to dst with the fewest links, as a list of devices from src to dst.

    Return [src] if src == dst, and None if dst can't be reached.
    """
    raise NotImplementedError("your turn")

In [ ]:
def describe(path, topology=WITH_ISOLATED):
    """Summarize a path so any correct answer passes: (devices on it, first, last, every hop is a real link)."""
    if path is None:
        return None
    links_ok = all(b in topology[a] for a, b in zip(path, path[1:]))
    return (len(path), path[0], path[-1], links_ok)

check(lambda s, d: describe(fewest_hops(WITH_ISOLATED, s, d)), [
    (("acc-sw-01", "edge-rtr"), (5, "acc-sw-01", "edge-rtr", True)),
    (("acc-sw-03", "acc-sw-01"), (5, "acc-sw-03", "acc-sw-01", True)),
    (("fw-01", "fw-01"), (1, "fw-01", "fw-01", True)),
    (("edge-rtr", "lab-sw-99"), None),
], name="fewest_hops")
check(fewest_hops, [((TOPOLOGY, "acc-sw-01", "acc-sw-02"), ["acc-sw-01", "dist-sw-01", "acc-sw-02"])]);

Two of those paths have more than one correct answer (there are two cores), so the check tests the
path's length and validity rather than one exact list. This is hop-count routing: what RIP computes.

### Exercise 12 — single points of failure

A design review question: which devices, if they fail, split the network? Brute force is the
expected approach: for each device, build the topology without it, flood from any remaining device
(your Exercise 10), and see whether everyone remaining still hears.

In [ ]:
def single_points_of_failure(topology: dict) -> set:
    """Devices whose removal leaves the remaining devices in more than one disconnected piece."""
    raise NotImplementedError("your turn")

In [ ]:
check(single_points_of_failure, [
    ((TOPOLOGY,), {"fw-01", "dist-sw-01", "dist-sw-02"}),
    (({"a": ["b"], "b": ["a", "c"], "c": ["b"]},), {"b"}),
    (({"a": ["b", "c"], "b": ["a", "c"], "c": ["a", "b"]},), set()),
]);

In [ ]:
try:
    seconds = best_of(lambda: single_points_of_failure(TOPOLOGY))
    print(f"brute force on {len(TOPOLOGY)} devices: {seconds * 1e6:.0f} µs")
except NotImplementedError:
    print("NOT YET (solve Exercise 12 to time it)")

Brute force is a legitimate first answer. Measure whether it's fast enough before making it clever.
For this network it takes microseconds. Note what it found: one firewall, and distribution switches
with single-homed access switches beneath them.

### Exercise 13 (stretch) — lowest latency, not fewest hops

Links now have a latency in ms. The flood no longer works, because a path with more hops can be
faster. Hint below.

In [ ]:
LATENCY = {   # (a, b) -> ms; links work in both directions
    ("edge-rtr", "fw-01"): 1.0,
    ("fw-01", "core-rtr-01"): 0.5, ("fw-01", "core-rtr-02"): 0.5,
    ("core-rtr-01", "core-rtr-02"): 0.2,
    ("core-rtr-01", "dist-sw-01"): 2.0, ("core-rtr-01", "dist-sw-02"): 0.4,
    ("core-rtr-02", "dist-sw-01"): 0.3, ("core-rtr-02", "dist-sw-02"): 0.4,
    ("dist-sw-01", "acc-sw-01"): 0.1, ("dist-sw-01", "acc-sw-02"): 0.1,
    ("dist-sw-02", "acc-sw-03"): 0.1,
}

<details><summary>Hint</summary>

`heapq` is a waiting line that always lets the smallest number out first. Put `(total_ms_so_far,
device, path)` in it; always continue from the cheapest one waiting; the first time `dst` comes out,
you're done.
</details>

In [ ]:
import heapq
line = []
for item in [(2.5, "b"), (0.4, "a"), (1.1, "c")]:
    heapq.heappush(line, item)
print(heapq.heappop(line), heapq.heappop(line))   # smallest first, whatever order they went in

In [ ]:
def lowest_latency_path(links: dict[tuple[str, str], float], src: str, dst: str):
    """Return (total_ms, path) for the path from src to dst with the lowest total latency,
    or None if dst can't be reached. Links work in both directions."""
    raise NotImplementedError("your turn")

In [ ]:
def rounded(result):
    return None if result is None else (round(result[0], 6), result[1])

check(lambda s, d: rounded(lowest_latency_path(LATENCY, s, d)), [
    (("acc-sw-01", "edge-rtr"), (1.9, ["acc-sw-01", "dist-sw-01", "core-rtr-02", "fw-01", "edge-rtr"])),
    (("acc-sw-03", "acc-sw-01"), (0.9, ["acc-sw-03", "dist-sw-02", "core-rtr-02", "dist-sw-01", "acc-sw-01"])),
    (("fw-01", "fw-01"), (0.0, ["fw-01"])),
    (("fw-01", "nowhere"), None),
], name="lowest_latency_path");

This is what link-state routing (OSPF, IS-IS) computes on every router.

<details><summary>What people call this</summary>

The flood with a <code>deque</code> is <b>breadth-first search</b> (BFS). Remembering who told whom builds a <b>shortest-path tree</b>.<br>
Removing each device and re-checking is <b>brute force</b>; the devices it finds are <b>articulation points</b> (cut vertices).<br>
<code>heapq</code> is a <b>priority queue</b> built on a <b>binary heap</b>; the lowest-latency search is <b>Dijkstra's algorithm</b>.<br>
Accumulate-then-sort is <b>aggregation</b> (a <b>group-by</b>).
</details>

## G. The reveal: pandas is your shapes

pandas is the standard library for tables of data. Build a DataFrame from the parsed rows: a list
of dicts, your §D row layout.

In [ ]:
import pandas as pd

df = pd.DataFrame(ROWS)
df.head()

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print(df.iloc[0].to_dict())          # one row, as a dict
print(df.to_dict("list")["bytes"])   # a dict of lists: your column layout

Side by side: your code and pandas computing the same thing. Where your exercise isn't solved yet, only the pandas side runs.

In [ ]:
def short(value, width=110):
    text = repr(value)
    return text if len(text) <= width else text[: width - 3] + "..."

def compare(label, yours, theirs):
    """Run your version and pandas's version; report whether they agree. Never fails."""
    theirs_value = theirs()
    try:
        yours_value = yours()
    except NotImplementedError:
        print(f"{label}: NOT YET\n  pandas: {short(theirs_value)}")
        return
    verdict = "same" if yours_value == theirs_value else "DIFFERENT"
    print(f"{label}: {verdict}\n  yours:  {short(yours_value)}\n  pandas: {short(theirs_value)}")

compare("rows_to_columns", lambda: rows_to_columns(ROWS), lambda: pd.DataFrame(ROWS).to_dict("list"))

In [ ]:
mask = df["action"] == "ALLOW"     # one True/False per row
print(mask.tolist())

compare("filter ALLOW rows",
        lambda: [r for r in ROWS if r["action"] == "ALLOW"],
        lambda: df[mask].to_dict("records"))

In [ ]:
compare("top_talkers",
        lambda: top_talkers(ROWS, 3),
        lambda: list(df.groupby("src_ip")["bytes"].sum().nlargest(3).items()))

In [ ]:
by_src = {}
for r in ROWS:                       # a dict from key to the rows with that key
    by_src.setdefault(r["src_ip"], []).append(r)

compare("lookup by src_ip",
        lambda: [r["bytes"] for r in by_src["10.0.0.5"]],
        lambda: df.set_index("src_ip").loc["10.0.0.5", "bytes"].tolist())

**The honest caveat.** Conceptually, a DataFrame is a dict of columns. Mechanically, each column is
not a Python list but a NumPy array: one contiguous block of same-typed numbers in memory, instead of
a list of pointers to separate Python objects. Measure what that buys.

In [ ]:
import numpy as np

col = df["bytes"].to_numpy()
print(type(col), col.dtype)

numbers_list = list(range(1_000_000))
numbers_array = np.arange(1_000_000)
t_list = best_of(lambda: sum(numbers_list))
t_array = best_of(lambda: numbers_array.sum())
print(f"sum of 1,000,000: Python list {t_list * 1e3:.2f} ms, NumPy array {t_array * 1e3:.2f} ms "
      f"({t_list / t_array:.0f}x)")

That gap is why pandas exists: your shapes and your walks, run over packed memory by compiled code.

### The library is on your disk

In [ ]:
import pandas
from pathlib import Path

pkg = Path(pandas.__file__).parent
print(pkg)
print(sorted(p.name for p in pkg.iterdir())[:20])
py_files = list(pkg.rglob("*.py"))
compiled = list(pkg.rglob("*.so")) + list(pkg.rglob("*.pyd"))
print(f"{len(py_files)} .py files, {len(compiled)} compiled extension files")

In [ ]:
import inspect

source = inspect.getsource(pandas.DataFrame.nlargest)
print(len(source.splitlines()), "lines; almost all docstring. The last line does the work:")
print(source.strip().splitlines()[-1])     # it hands off to more pandas code: libraries call libraries

In [ ]:
from pandas.core import common
print(inspect.getsource(common.count_not_none))   # a whole pandas function: Python you can already read

In [ ]:
for p in compiled[:8]:
    print(p.relative_to(pkg))

The `.so`/`.pyd` files are compiled C and Cython: the fast inner loops. The rest is Python, just
like yours. `import` means "run code from a folder on this machine." Notebook 02 starts there.

<details><summary>What people call this</summary>

A <b>DataFrame</b> is a table: <b>columns</b> of a single <b>dtype</b> each, plus an <b>index</b> (row labels).<br>
A column of True/False used to select rows is a <b>boolean mask</b>; selecting with it is <b>filtering</b>.<br>
One contiguous block of same-typed numbers is an <b>array</b>; operating on all of it at once in compiled code is <b>vectorization</b>.
</details>

## What you can do now

- Hold network data in lists, tuples, dicts, sets and dataclasses, and choose between them by need.
- Convert addresses to numbers, test subnet membership, match firewall rules, look up routes.
- Walk a topology: who's reachable, the fewest-hop path, the single points of failure.
- Measure instead of guess, and read a traceback from the bottom up.
- See that pandas is these same shapes and walks, on your disk.

## Glossary: what people call these

| You did | People call it |
|---|---|
| text, whole numbers, measurements, yes/no, nothing | `str`, `int`, `float`, `bool`, `None`: primitive types |
| `&`, `\|`, `<<`, `>>` | bitwise operators, bit masking |
| ordered sequence; walking it | list (dynamic array); linear search, O(n) |
| fixed record | tuple; immutable |
| find by name instantly | dict; hash map, O(1) lookup; keys must be hashable |
| membership and overlap | set; intersection, union, difference |
| most specific prefix wins | longest prefix match |
| rows vs columns | row-oriented vs column-oriented layout |
| device → neighbors | graph, adjacency list, nodes and edges |
| same-end add/remove; arrival-order line | stack (LIFO); queue (FIFO); deque |
| dict with a contract | class, instance, attribute, method, schema |
| flooding; who told whom | breadth-first search; shortest-path tree |
| remove each and re-check | brute force; articulation points |
| smallest-first waiting line; lowest latency | priority queue (heap); Dijkstra's algorithm |
| total per source, sorted | aggregation, group-by |
| packed same-typed column | array, vectorization |

## Commit checkpoint

Save your work to git, in a terminal at the repo root:

```bash
git status                                   # see what changed
git add notebooks/01_shapes_and_walking_them.ipynb
git commit -m "notebook 01: exercises"
```

A commit is a snapshot you can return to. Outputs are stripped automatically on commit (nbstripout
was installed when this repo was built), so only your code goes into history. Notebook 02 explains why.